# types_jsonschema smoke test

`types-jsonschema` is a PEP 561 type stub package for `jsonschema`. It installs `.pyi` files into a `jsonschema-stubs` directory so type checkers (mypy, pyright) can statically check code that uses `jsonschema`. There is no runtime module to import.

In [ ]:
# Submodule smoke test for a PEP 561 stub package.
# types-jsonschema installs a `jsonschema-stubs` directory (not a regular import target).
# We locate it via importlib.metadata and walk its contents.
import importlib.metadata as im
import pathlib

dist = im.distribution('types-jsonschema')
print(f"types-jsonschema version = {dist.version}")

stub_files = [f for f in (dist.files or []) if str(f).endswith('.pyi')]
print(f"Stub file count: {len(stub_files)}")
for f in stub_files[:20]:
    print(f"  {f}")

assert len(stub_files) > 0, 'expected at least one .pyi stub file in types-jsonschema'

In [ ]:
# Example 1: locate the jsonschema-stubs directory on disk.
import importlib.metadata as im
import pathlib

dist = im.distribution('types-jsonschema')
stubs_root = None
for f in dist.files or []:
    parts = pathlib.Path(str(f)).parts
    if 'jsonschema-stubs' in parts:
        idx = parts.index('jsonschema-stubs')
        candidate = dist.locate_file(pathlib.Path(*parts[: idx + 1]))
        stubs_root = pathlib.Path(str(candidate))
        break

print(f"jsonschema-stubs directory: {stubs_root}")
assert stubs_root is not None, 'jsonschema-stubs directory not found'
assert stubs_root.is_dir(), f'expected directory at {stubs_root}'

In [ ]:
# Example 2: verify the main __init__.pyi stub exists and is non-empty.
import importlib.metadata as im
import pathlib

dist = im.distribution('types-jsonschema')
init_pyi = None
for f in dist.files or []:
    if str(f).endswith('jsonschema-stubs/__init__.pyi'):
        init_pyi = pathlib.Path(str(dist.locate_file(f)))
        break

print(f"__init__.pyi: {init_pyi}")
assert init_pyi is not None and init_pyi.is_file()
content = init_pyi.read_text()
print(f"size = {len(content)} bytes; first 200 chars:")
print(content[:200])
assert len(content) > 0, 'jsonschema-stubs/__init__.pyi is empty'

In [ ]:
# Example 3: parse every stub file with the standard `ast` module to confirm valid Python syntax.
import ast
import importlib.metadata as im
import pathlib

dist = im.distribution('types-jsonschema')
parsed = 0
failed = []
for f in dist.files or []:
    if str(f).endswith('.pyi'):
        path = pathlib.Path(str(dist.locate_file(f)))
        try:
            ast.parse(path.read_text(), filename=str(path), type_comments=True)
            parsed += 1
        except SyntaxError as e:
            failed.append((str(f), str(e)))

print(f"Successfully parsed {parsed} .pyi files")
if failed:
    for name, err in failed:
        print(f"  FAILED {name}: {err}")
assert parsed > 0 and not failed

In [ ]:
# Example 4: confirm key jsonschema public-API symbols are declared in the stubs.
import importlib.metadata as im
import pathlib

dist = im.distribution('types-jsonschema')
init_pyi = next(
    pathlib.Path(str(dist.locate_file(f)))
    for f in (dist.files or [])
    if str(f).endswith('jsonschema-stubs/__init__.pyi')
)
text = init_pyi.read_text()

expected_symbols = ['validate', 'ValidationError', 'SchemaError', 'Draft7Validator', 'FormatChecker']
found = {sym: (sym in text) for sym in expected_symbols}
for sym, present in found.items():
    print(f"  {sym}: {'yes' if present else 'NO'}")

assert all(found.values()), f'missing symbols in stubs: {found}'

In [ ]:
# Example 5: verify the distribution metadata advertises this as a PEP 561 stub package
# (Typing :: Stubs Only classifier) and the dist-info version matches the installed version.
import importlib.metadata as im

dist = im.distribution('types-jsonschema')
meta = dist.metadata

print(f"Name: {meta['Name']}")
print(f"Version: {meta['Version']}")
assert dist.version == im.version('types-jsonschema'), 'dist-info version mismatch'
summary = meta['Summary'] or ''
print(f"Summary: {summary}")

classifiers = meta.get_all('Classifier') or []
typing_classifiers = [c for c in classifiers if 'Typing' in c or 'Stubs' in c]
print(f"Typing classifiers: {typing_classifiers}")

is_stub_pkg = (
    meta['Name'].lower().startswith('types-')
    or 'stub' in summary.lower()
    or any('Stubs' in c for c in classifiers)
)
assert is_stub_pkg, 'types-jsonschema does not look like a PEP 561 stub package'
print('\nAll checks passed: types-jsonschema is a valid PEP 561 stub package.')